# RAG FUSION Pipeline
### PDF → Chunking → FAISS → Query Variations → Retrieval → Reciprocal Rank Fusion → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [3]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Embeddings & Store in FAISS

In [5]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Generate Query Variations using LLM

In [6]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

query = "What role did Abdul Kalam play in India's missile program?"

variation_prompt = f"""Generate 3 different versions of the following question to help retrieve relevant documents.
Each version should approach the question from a different angle.
Return only the 3 questions, one per line.

Original question: {query}"""

response = llm.invoke(variation_prompt)
query_variations = [q.strip() for q in response.content.strip().split("\n") if q.strip()]

# Include original query
all_queries = [query] + query_variations

print(f"Original: {query}")
print(f"\nGenerated Variations:")
for i, q in enumerate(query_variations, 1):
    print(f"  {i}. {q}")

Original: What role did Abdul Kalam play in India's missile program?

Generated Variations:
  1. 1. What was the significance of Abdul Kalam's involvement in India's defense technology development?
  2. 2. How did Abdul Kalam's expertise contribute to the success of India's indigenous missile program?
  3. 3. What was the impact of Abdul Kalam's leadership on India's strategic missile capabilities?


## Step 5: Retrieve Documents for Each Query Variation

In [7]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

all_ranked_results = []
for q in all_queries:
    docs = retriever.invoke(q)
    all_ranked_results.append(docs)

print(f"Queries used: {len(all_queries)}")
print(f"Documents per query: {len(all_ranked_results[0])}")
print(f"Total retrievals: {sum(len(r) for r in all_ranked_results)}")

Queries used: 4
Documents per query: 5
Total retrievals: 20


## Step 6: Reciprocal Rank Fusion (RRF)

In [8]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    rrf_scores = {}
    doc_map = {}
    
    for ranked_docs in ranked_lists:
        for rank, doc in enumerate(ranked_docs, 1):
            doc_id = doc.page_content
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0
                doc_map[doc_id] = doc
            rrf_scores[doc_id] += 1 / (k + rank)
    
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(doc_map[doc_id], score) for doc_id, score in sorted_docs]

fused_results = reciprocal_rank_fusion(all_ranked_results)

# Pick top 3
top_k = 3
fused_docs = [doc for doc, _ in fused_results[:top_k]]

print(f"Fused Top {top_k} Documents:")
for i, (doc, score) in enumerate(fused_results[:top_k]):
    print(f"\n--- Fused {i+1} | RRF Score: {score:.4f} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:200])

Fused Top 3 Documents:

--- Fused 1 | RRF Score: 0.0653 (Page 4) ---
US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kalam as its chief executive.[28] Kalam played a major role in the development
of m

--- Fused 2 | RRF Score: 0.0648 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, ma

--- Fused 3 | RRF Score: 0.0632 (Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him re


## Step 7: Query LLM with Fused Context

In [9]:
# Build context from fused chunks
context = "\n\n".join([doc.page_content for doc in fused_docs])

# Create prompt
prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""

# Get response
response = llm.invoke(prompt)
print("Answer:", response.content)

Answer: Abdul Kalam played a major role in the development of India's missiles, including Agni and Prithvi, and was known as the "Missile Man of India" for his work on ballistic missile and launch vehicle technology.


## Fused Source Chunks

In [10]:
for i, doc in enumerate(fused_docs):
    print(f"\n--- Chunk {i+1} (Page {doc.metadata['page'] + 1}) ---")
    print(doc.page_content[:300])


--- Chunk 1 (Page 4) ---
US$780 million in 2023) for the project titled Integrated Guided Missile Development Programme
(IGMDP) and appointed Kalam as its chief executive.[28] Kalam played a major role in the development
of missiles including Agni, an intermediate range ballistic missile and Prithvi, the tactical surface-to

--- Chunk 2 (Page 1) ---
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a
scientist and science administrator, mainly at the
Defence Research and Development Organisation
(DRDO) and Indian Space Research Organisat

--- Chunk 3 (Page 4) ---
Kalam greeting then prime minister
Vajpayee on 25 December 2003
major role in convincing the cabinet to conceal the true nature of these classified projects. His research
and leadership brought him recognition in the 1980s, which prompted the government to initiate an
advanced missile programme unde


Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?